# Movie genre classifier

##### This notebook contains an implementation of my naive bayes text classifier to classify several movies into their genres based upon a plot description.

___

Import pandas for analysis using dataframes

In [26]:
import pandas as pd

I will be reading from a csv dataset I found on Kaggle which contains movie's wikipedia data, shown below.

In [27]:
movies_df = pd.read_csv('data\wiki_movie_plots_deduped.csv')
movies_df.head()

,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot
0,1901,Kansas Saloon Smashers,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Kansas_Saloon_Sm...,"A bartender is working at a saloon, serving dr..."
1,1901,Love by the Light of the Moon,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Love_by_the_Ligh...,"The moon, painted with a smiling face hangs ov..."
2,1901,The Martyred Presidents,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/The_Martyred_Pre...,"The film, just over a minute long, is composed..."
3,1901,"Terrible Teddy, the Grizzly King",American,Unknown,NaN,unknown,"https://en.wikipedia.org/wiki/Terrible_Teddy,_...",Lasting just 61 seconds and consisting of two ...
4,1902,Jack and the Beanstalk,American,"George S. Fleming, Edwin S. Porter",NaN,unknown,https://en.wikipedia.org/wiki/Jack_and_the_Bea...,The earliest known adaptation of the classic f...


The data needs to only contain the genre and plot as that is what will be used to train the model. Because some movies had multiple genres, I seperated them and duplicated the plot so that it appears for all corresponding entries.

In [28]:
genre_description = (
    movies_df[['Genre', 'Plot']]
    .assign(Genre=movies_df['Genre'].str.split(r'[/,\-]'))
    .explode('Genre')
    .reset_index(drop=True)
)
genre_description.head(20)

,Genre,Plot
0,unknown,"A bartender is working at a saloon, serving dr..."
1,unknown,"The moon, painted with a smiling face hangs ov..."
2,unknown,"The film, just over a minute long, is composed..."
3,unknown,Lasting just 61 seconds and consisting of two ...
4,unknown,The earliest known adaptation of the classic f...
5,unknown,"Alice follows a large white rabbit down a ""Rab..."
6,western,The film opens with two bandits breaking into ...
7,comedy,The film is about a family who move to the sub...
8,unknown,The opening scene shows the interior of the ro...
9,unknown,Scenes are introduced using lines of the poem....


Since we are classifying by genre, there can't be too many otherwise the predictions will be less precise so I analysed the genres present currently in the dataset.

In [29]:
genres = genre_description['Genre'].unique()
print(genres)

['unknown' 'western' 'comedy' ... ' fantasy film' 'ero'
 'horror romantic comedy']


From analysing these genres, I realised that too many of them are vague and can mean several others, there are genres which can fit inside others and there are ones which don't mean anything because of errors with the previous code. Because of this I have chosen common genres which can be used as the categories for classification. Genres which don't mean anything are dropped and ones which fit inside others are mapped to them via a genre map as shown below.

In [30]:
final_genres = [
    'action', 'adventure', 'animation', 'biographical', 'comedy', 'crime',
    'documentary', 'drama', 'family', 'fantasy', 'historical', 'horror',
    'musical', 'mystery', 'romance', 'sci-fi', 'sports', 'thriller', 'war', 'western'
]

genre_map = {
    'sci fi': 'sci-fi',
    'sciencefiction': 'sci-fi',
    'science-fiction': 'sci-fi',
    'romcom': 'romance',
    'romantic-comedy': 'romance',
    'comedy-drama': 'comedy',
    'action-comedy': 'action',
    'action adventure': 'action',
    'fantasy comedy': 'fantasy',
    'drama-romance': 'drama',
    'horror-comedy': 'horror',
    'fantasy-adventure': 'fantasy',
    'animated-adventure': 'animation',
    'musical-comedy': 'musical',
    'biopic': 'biographical',
    'documentary-biographical': 'documentary',
    'psychological thriller': 'thriller',
    'science-fantasy': 'fantasy',
    'family-comedy': 'family'
}

genre_description['Genre'] = genre_description['Genre'].str.strip().str.lower()
genre_description = genre_description[
    (genre_description['Genre'] != '') &
    (genre_description['Genre'] != 'unknown')
]
genre_description['Genre'] = genre_description['Genre'].replace(genre_map)
genre_description = genre_description[
    genre_description['Genre'].isin(final_genres)
].reset_index(drop=True)
genre_description.head()

,Genre,Plot
0,western,The film opens with two bandits breaking into ...
1,comedy,The film is about a family who move to the sub...
2,biographical,Boone's daughter befriends an Indian maiden as...
3,comedy,Before heading out to a baseball game at a nea...
4,comedy,The plot is that of a black woman going to the...


Now that the genres are sorted, stopwords need to be removed from the plots as they will create noise from appearing in every plot. I will be using the stopwords corpus from nltk.

In [31]:
import nltk
nltk.download('stopwords')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jack0\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

Shown below are the stopwords which I am removing from the plots.

In [32]:
from nltk.corpus import stopwords

stopwords = set(stopwords.words('english'))
print(stopwords)

{'what', 'each', 'it', 'theirs', 'my', 'a', 'further', 'll', 'any', 'such', "haven't", 'she', 'of', 'then', 'down', 'than', 'not', "mustn't", 'be', 'didn', 'below', "they'll", 'to', 'why', 'with', 'yourselves', 'between', 't', 'once', "they're", "he'd", "needn't", 'above', "don't", "it'd", 'have', 'same', "we'll", 'y', 'up', 'other', 'who', "weren't", "should've", "she's", 'all', 'having', 'whom', 'herself', 'after', 'has', "aren't", 'as', "it'll", "we've", 'yours', 'haven', 'am', 'did', "i'll", "mightn't", 'its', "it's", 'itself', 'hadn', "shan't", 'those', "didn't", 'if', 'under', "won't", "that'll", 'both', 'so', 'themselves', "you'd", 'before', 'being', "they'd", 'wasn', 'or', 'them', 'won', 'doing', "isn't", 'their', 'we', 'during', 'is', 'o', "we're", 'been', 'again', 'he', 'couldn', 'nor', 'more', 'ours', "hadn't", 're', 'into', "he's", "she'll", 'few', 've', 'ma', 'shouldn', "couldn't", "he'll", 'himself', 'needn', 'your', 'where', 'i', 'over', 'out', 'against', 'our', 'only', 

There are other film related words I noticed in most of the plots other than these words which I shall be also removing.

In [33]:
stopwords.add('film')
stopwords.add('features')

In order to clean the text to train the model, the plots need to be in lowercase, include no punctuation and have no stopwords. I wrote this function to do exactly that.

In [34]:
def clean_text(text):
    #Lowercase
    text = text.lower()
    #Split up words
    words = text.split()
    #Remove stopwords
    words = [w for w in words if w not in stopwords]
    #Remove punctuation
    words = [w for w in words if w.isalpha()]
    #Put string back together
    string = ' '.join(words)

    return string

The function is applied to each plot in the dataframe to leave a cleaned dataset.

In [35]:
processed = genre_description.copy()
processed['Plot'] = processed['Plot'].apply(clean_text)
processed.head()

,Genre,Plot
0,western,opens two bandits breaking railroad telegraph ...
1,comedy,family move hoping quiet things start go wife ...
2,biographical,daughter befriends indian maiden boone compani...
3,comedy,heading baseball game nearby sports fan brown ...
4,comedy,plot black woman going dentist toothache given...


Now the data is preprocessed, it needs splitting into training data and testing data. I will be using the common 80%, 20% split.

In [36]:
#Shuffle data
processed = processed.sample(frac=1, random_state=10)

#Split data
split_size = int(len(processed) * 0.8)
train_data = processed.iloc[:split_size]
test_data = processed.iloc[split_size:]

Now the model needs to be trained. I will be using my own naive bayes classifier as shown below.

In [37]:
from bayes_classifier import TextClassifier

The model is instantiated with the priors and likelihoods automatically calculated.

In [38]:
genre_classifier = TextClassifier(train_data)

Add column into test data for each prediction

In [39]:
test_data['Prediction'] = genre_classifier.predict(test_data['Plot'])

C:\Users\jack0\AppData\Local\Temp\ipykernel_28652\199233873.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_data['Prediction'] = genre_classifier.predict(test_data['Plot'])


Analyse which predictions are correct

In [40]:
correct = test_data[test_data['Genre'] == test_data['Prediction']]
print(len(correct) / len(test_data))

0.4184200812107789
